<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec08_resampling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec08_resampling.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第8回 幾何処理（拡大縮小）
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，SciPy，scikit-image，Matplotlib

MATLAB のライブスクリプト `code/matlab/vie_sec08_resampling.m` を Python に移植したもの。第8回のスライド（vie2026-08）で使う図と数値を計算する。間引き・平均による縮小，最近傍・双線形補間による拡大，標本化と周波数スペクトル（一次元と多次元），ダウン／アップサンプラ，デシメーションとインタポレーション（エリアシングとイメージング），補間フィルタ（零次ホールド，双線形，3次畳み込み補間）を扱う。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

記号は教科書に合わせる。再標本化行列を $\boldsymbol{M}$（一次元では間引き率・補間率 $M$），標本化行列を $\boldsymbol{L}$，一次元の標本化間隔を $\Delta_\mathrm{t}$，二次元の標本化間隔を $\Delta_\mathrm{v},\Delta_\mathrm{h}$（垂直・水平），連続の角周波数を $\nu$，正規化角周波数を $\omega$ とする。波形 $u(\cdot)$ のフーリエ変換を $\tilde{u}(\cdot)$，サンプル列（櫛状化した分布）$x(\cdot)$ のフーリエ変換を $\tilde{x}(\cdot)$，配列の DSFT を $X(\mathrm{e}^{\mathrm{j}\omega})$ と書く。

画像処理には scikit-image，SciPy，NumPy を使う。教科書の例題と小さな配列の例の数値はスライドと一致することを `assert` で確かめている。画像の処理結果は，ライブラリの実装が MATLAB と異なるため，スライドの図とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from scipy import ndimage, signal
from skimage.util import img_as_float
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の配色はスライドと揃え，ロゴの 3 色（メインの緑，寒色系の青，暖色系の橙）と灰色を使う
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

参考資料のサンプル画像 msipimg06（縞模様の路面）をグレースケールにし，$256\times256$ に縮小して使う。段の縁の細い溝（1 画素ほどの幅の斜めの線）と石の細かい模様が，間引きでエリアシングを起こしやすい（溝が途切れた破線に化ける）。msipimg07（モンブラン），msipimg08（スイカ）の切り出しとも比べ，エリアシングが最もはっきり見えるこの画像を選んだ。

In [ ]:
X = img_as_float(vie.msipimg(6, 256, "gray"))   # 256×256 のグレースケール
print("大きさ:", X.shape)

## 間引きによる縮小
垂直・水平とも偶数番目の画素だけを残す（$y[\boldsymbol{n}]=x[\boldsymbol{M}\boldsymbol{n}]$，$\boldsymbol{M}=\mathrm{diag}(2,2)$）。柵の縞が別の模様に化ける（エリアシング）。

In [ ]:
Yd = X[::2, ::2]

## 平均による縮小
例：$2\times2$ ブロックの平均値を出力する（教科書の例「ブロック平均」）。

In [ ]:
x4 = np.array([[0, 2, 4, 4], [4, 6, 4, 4], [0, 6, 4, 4], [2, 4, 6, 6]])
y2 = (x4[0::2, 0::2] + x4[1::2, 0::2] + x4[0::2, 1::2] + x4[1::2, 1::2]) / 4
print("x4 =\n", x4)
print("y2 =\n", y2)
assert np.array_equal(y2, [[3, 4], [3, 5]])   # スライドの値と一致

スライドの図（TikZ）のマス目の数値も，ここから書き出す。左上のブロックの平均の計算式も添える。

In [ ]:
blk = x4[:2, :2].ravel()                       # x[0,0], x[0,1], x[1,0], x[1,1] の順
print("({}+{}+{}+{})/4 = {:g}".format(*blk, y2[0, 0]))

画像にも施す。

In [ ]:
Ya = (X[0::2, 0::2] + X[1::2, 0::2] + X[0::2, 1::2] + X[1::2, 1::2]) / 4
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], X, "原画像")
show(axs[1], Yd, "間引き")
show(axs[2], Ya, "ブロック平均")
plt.show()

## 最近傍補間による拡大
各画素を $2\times2$ ブロックに複製する。

In [ ]:
y8nn = np.kron(x4, np.ones((2, 2), dtype=int))
print("y8nn =\n", y8nn)
assert np.array_equal(y8nn, x4.repeat(2, axis=0).repeat(2, axis=1))
Znn = np.kron(Ya, np.ones((2, 2)))             # 縮小画像を 2×2 倍に拡大

## 双線形補間による拡大
零値挿入の後，線形補間フィルタ $f[n]=(\frac12,1,\frac12)$ を垂直・水平に施す（周囲は零値）。元の画素はそのまま残り，間は隣どうしの平均で埋まる。

In [ ]:
fl = np.array([1 / 2, 1, 1 / 2])
u8 = np.zeros((8, 8))
u8[::2, ::2] = x4                              # 零値挿入
y8bl = signal.convolve2d(u8, np.outer(fl, fl), "full")
y8bl = y8bl[1:9, 1:9]                          # 元の画素が偶数番目に来るように切り出す
np.set_printoptions(linewidth=120)
print("y8bl =\n", y8bl)
y8bl_slide = np.array([[0, 1, 2, 3, 4, 4, 4, 2],
                       [2, 3, 4, 4, 4, 4, 4, 2],
                       [4, 5, 6, 5, 4, 4, 4, 2],
                       [2, 4, 6, 5, 4, 4, 4, 2],
                       [0, 3, 6, 5, 4, 4, 4, 2],
                       [1, 3, 5, 5, 5, 5, 5, 2.5],
                       [2, 3, 4, 5, 6, 6, 6, 3],
                       [1, 1.5, 2, 2.5, 3, 3, 3, 1.5]])
assert np.array_equal(y8bl, y8bl_slide)        # スライドの値と一致

スライドの例で取り上げる 3 つの値。$y[0,1]$ は水平に隣り合う 2 画素の平均，$y[1,1]$ は 4 画素の平均，$y[7,7]$ は右下の隅で，元の画素 $x[3,3]$ と外側の零値 3 つの平均になる。

In [ ]:
assert (y8bl[0, 1] == (x4[0, 0] + x4[0, 1]) / 2 and y8bl[1, 1] == x4[:2, :2].sum() / 4
        and y8bl[7, 7] == x4[3, 3] / 4)
print(f"y[0,1] = ({x4[0, 0]}+{x4[0, 1]})/2 = {y8bl[0, 1]:g}")
print("y[1,1] = ({}+{}+{}+{})/4 = {:g}".format(*blk, y8bl[1, 1]))
print(f"y[7,7] = {x4[3, 3]}/4 = {y8bl[7, 7]:g}")

画像にも施す（境界は複製拡張）。

In [ ]:
Ub = np.zeros((2 * Ya.shape[0], 2 * Ya.shape[1]))
Ub[::2, ::2] = Ya
Zbl = ndimage.correlate(Ub, np.outer(fl, fl), mode="nearest")
Zbl[:, -1] = Zbl[:, -2]                        # 右端・下端は複製で補う
Zbl[-1, :] = Zbl[-2, :]
fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
show(axs[0], Znn, "最近傍補間")
show(axs[1], Zbl, "双線形補間")
plt.show()

拡大部分（青い帯の斜めの端を含む $64\times64$ の領域）を比べる。最近傍補間では斜めの縁が階段状になり，双線形補間ではなめらかになる。

In [ ]:
zr, zc = slice(119, 183), slice(49, 113)       # 斜めの縁を含む領域
fig, axs = plt.subplots(1, 2, figsize=(8, 4), layout="constrained")
show(axs[0], Znn[zr, zc], "最近傍補間（拡大部分）")
show(axs[1], Zbl[zr, zc], "双線形補間（拡大部分）")
plt.show()

## 一次元の標本化とスペクトル
波形 $u(t)$ を標本化間隔 $\Delta_\mathrm{t}$ で標本化する（$D=1$，$\boldsymbol{L}=\Delta_\mathrm{t}$）。サンプル列 $x(t)=u(t)\,\mathrm{comb}_{\Delta_\mathrm{t}}(t)$ のフーリエ変換は

$$\tilde{x}(\nu)=\frac{1}{\Delta_\mathrm{t}}\sum_{k=-\infty}^{\infty}\tilde{u}(\nu-k\nu_\mathrm{s}),\quad \nu_\mathrm{s}=\frac{2\pi}{\Delta_\mathrm{t}}$$

となり，スペクトル $\tilde{u}(\nu)$ が $\nu_\mathrm{s}$ ごとに周期化される（教科書 4.1.3 項）。帯域が $|\nu|<\nu_\mathrm{max}$ のとき，$\Delta_\mathrm{t}$ が大きく $\nu_\mathrm{s}<2\nu_\mathrm{max}$ となると隣どうしが重なってエリアシングが生じる。縦軸は $\Delta_\mathrm{t}$ 倍した $\Delta_\mathrm{t}\tilde{x}(\nu)$ で描く。

In [ ]:
nuv = np.linspace(-5, 5, 2001)                 # 角周波数 ν（単位は ν_max）


def tri(v):
    """三角形のスペクトル ũ(ν)（帯域 ν_max）"""
    return np.maximum(1 - np.abs(v), 0)


nusList = [3, 1.4]                             # 標本化角周波数 ν_s = 2π/Δ_t（ν_max の倍数）
fig, axs = plt.subplots(3, 1, figsize=(6, 5.2), sharex=True, layout="constrained")
axs[0].plot(nuv, tri(nuv), color=ccool, lw=2)
axs[0].set_title(r"$\tilde{u}(\nu)$")
axs[0].text(-4.9, 1.2, "原信号のスペクトル", color=ccool, va="center", fontsize=9)
for ax, nus in zip(axs[1:], nusList):
    S = np.zeros_like(nuv)
    for k in range(-5, 6):
        ax.plot(nuv, tri(nuv - k * nus), ":", color=cgray)
        S = S + tri(nuv - k * nus)             # Δ_t x̃(ν) = Σ_k ũ(ν - kν_s)
    if nus >= 2:
        cS, note = cmain, "重ならない"
    else:
        cS, note = cwarm, "重なる：エリアシング"
    ax.plot(nuv, S, color=cS, lw=2)
    ax.annotate("", xy=(nus, 1.2), xytext=(0, 1.2),                 # ν_s の間隔
                arrowprops=dict(arrowstyle="<|-|>", color=cgray, shrinkA=0, shrinkB=0))
    ax.text(nus + 0.1, 1.2, r"$\nu_\mathrm{s}$", ha="left", va="center")
    ax.text(-4.9, 1.2, note, color=cS, va="center", fontsize=9)
    ax.set_title(rf"$\Delta_\mathrm{{t}}\tilde{{x}}(\nu)\quad(\nu_\mathrm{{s}}=2\pi/\Delta_\mathrm{{t}}={nus:g}\nu_\mathrm{{max}})$")
axs[-1].set_xlabel(r"$\nu/\nu_\mathrm{max}$")
for ax in axs:
    ax.set(ylim=(0, 1.5), yticks=[0, 1], xticks=np.arange(-5, 6), xlim=(-5, 5))
    ax.grid(True)
plt.show()

## 多次元の標本化とスペクトル
二次元では標本化行列 $\boldsymbol{L}=\mathrm{diag}(\Delta_\mathrm{v},\Delta_\mathrm{h})$ の格子で標本化する。サンプル列のフーリエ変換は

$$\tilde{x}(\boldsymbol{\nu})=\frac{1}{|\det(\boldsymbol{L})|}\sum_{\boldsymbol{m}\in\mathbb{Z}^D}\tilde{u}(\boldsymbol{\nu}-2\pi\boldsymbol{L}^{-\top}\boldsymbol{m})$$

となり，周期構造行列 $2\pi\boldsymbol{L}^{-\top}$ の周期性をもつ（教科書 4.1.3 項）。標本化格子と，標本化後の周波数スペクトルのサポート（台）を描く。垂直の間隔を水平の 2 倍にとる。

In [ ]:
Dv, Dh = 2, 1                                  # 標本化間隔（垂直・水平）
L = np.diag([Dv, Dh])                          # 標本化行列（第 1 次元が垂直）
detL = abs(np.linalg.det(L))                   # 基本平行多面体 FPD(L) の面積
Lp = 2 * np.pi * np.linalg.inv(L.T)            # 周期構造行列 2πL^{-T}
print("L =\n", L)
print("|det(L)| =", f"{detL:g}")
print("2πL^{-T} =\n", Lp)


def pistr(v):
    """π の整数倍を文字列にする（例：2π → "2\\pi"，π → "\\pi"，0 → "0"）"""
    c = round(v / np.pi, 6)
    if c == 0:
        return "0"
    if c == 1:
        return r"\pi"
    if c == -1:
        return r"-\pi"
    return f"{c:g}" + r"\pi"


print("2πL^{-T} の対角成分（垂直・水平の周期）:", pistr(Lp[0, 0]), ",", pistr(Lp[1, 1]))
assert (Dv, Dh, detL) == (2, 1, 2) and pistr(Lp[0, 0]) == r"\pi" and pistr(Lp[1, 1]) == r"2\pi"
assert Lp[0, 1] == 0 and Lp[1, 0] == 0         # スライドの値（diag(π, 2π)）と一致

標本化格子 $\boldsymbol{q}=\boldsymbol{L}\boldsymbol{n}$。教科書の図と同じく，水平 $q_\mathrm{h}$ を右向き，垂直 $q_\mathrm{v}$ を下向きにとる。薄い緑の領域が基本平行多面体 $\mathrm{FPD}(\boldsymbol{L})$（面積 $|\det(\boldsymbol{L})|$），矢印が $\boldsymbol{L}$ の列ベクトル。

In [ ]:
nh, nv = np.meshgrid(np.arange(5), np.arange(3))
Q = L @ np.vstack([nv.ravel(), nh.ravel()])    # 標本点（第 1 行が q_v，第 2 行が q_h）
fig, ax = plt.subplots(figsize=(3.6, 3.7), layout="constrained")
ax.fill([0, Dh, Dh, 0], [0, 0, Dv, Dv], color=cmain, alpha=0.15, lw=0)
ax.plot(Q[1], Q[0], "o", ms=6, mfc=cmain, mec=cmain)
arrow = dict(arrowstyle="-|>", color=cwarm, lw=1.5, shrinkA=0, shrinkB=0, mutation_scale=12)
ax.annotate("", xy=(0, Dv), xytext=(0, 0), arrowprops=arrow)
ax.annotate("", xy=(Dh, 0), xytext=(0, 0), arrowprops=arrow)
ax.text(-0.12, Dv * 0.55, r"$\Delta_\mathrm{v}$", ha="right", va="center", color=cwarm)
ax.text(Dh * 0.5, 0.12, r"$\Delta_\mathrm{h}$", ha="center", va="top", color=cwarm)
ax.set(xticks=range(5), yticks=range(5), xlim=(-0.7, 4.5), ylim=(4.6, -0.4), aspect="equal",
       xlabel="$q_\\mathrm{h}$", ylabel="$q_\\mathrm{v}$")
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")
ax.grid(True)
for sp in ("right", "bottom"):
    ax.spines[sp].set_visible(False)
plt.show()

波形のスペクトル $\tilde{u}(\boldsymbol{\nu})$ のサポートをひし形 $|\nu_\mathrm{v}|/(0.45\pi)+|\nu_\mathrm{h}|/(0.9\pi)\le1$ とする。標本化後は $2\pi\boldsymbol{L}^{-\top}\boldsymbol{m}$ だけずれた複製が並ぶ（青：元のサポート，緑：複製）。破線は複製ごとの周期の区切り（$\boldsymbol{0}$ を中心とする $\mathrm{FPD}(2\pi\boldsymbol{L}^{-\top})$ の平行移動）で，サポートが区切りの中に収まれば重ならない。矢印が $2\pi\boldsymbol{L}^{-\top}$ の列ベクトル（垂直の周期 $2\pi/\Delta_\mathrm{v}$，水平の周期 $2\pi/\Delta_\mathrm{h}$）。

In [ ]:
av, ah = 0.45 * np.pi, 0.9 * np.pi             # サポートの半径（垂直・水平）
sup = np.array([[0, -av], [ah, 0], [0, av], [-ah, 0]])   # 頂点（横軸 ν_h，縦軸 ν_v）
lim = 2.5 * np.pi
fig, ax = plt.subplots(figsize=(4.2, 4.2), layout="constrained")
for mv in range(-4, 5):
    for mh in range(-2, 3):
        s = Lp @ [mv, mh]                      # ずれ 2πL^{-T} m（第 1 成分が垂直）
        if mv == 0 and mh == 0:
            continue
        ax.add_patch(Polygon(sup + [s[1], s[0]], fc=(*cmain, 0.25), ec=cmain))
ax.add_patch(Polygon(sup, fc=(*ccool, 0.7), ec=ccool))
for v in np.arange(-3, 4, 2) * Lp[1, 1] / 2:   # 周期の区切り（水平）
    ax.axvline(v, ls="--", color=cgray)
for v in np.arange(-5, 6, 2) * Lp[0, 0] / 2:   # 周期の区切り（垂直）
    ax.axhline(v, ls="--", color=cgray)
ax.axvline(0, color="k", lw=0.8)
ax.axhline(0, color="k", lw=0.8)
arrow = dict(arrowstyle="-|>", color=cwarm, lw=1.5, shrinkA=0, shrinkB=0, mutation_scale=12)
ax.annotate("", xy=(Lp[1, 1], 0), xytext=(0, 0), arrowprops=arrow)
ax.annotate("", xy=(0, Lp[0, 0]), xytext=(0, 0), arrowprops=arrow)
bbox = dict(fc="w", ec="none", pad=0.5)
ax.text(Lp[1, 1], -0.12 * np.pi, r"$2\pi/\Delta_\mathrm{h}$", ha="center", va="bottom", color=cwarm, bbox=bbox)
ax.text(0.1 * np.pi, Lp[0, 0] + 0.1 * np.pi, r"$2\pi/\Delta_\mathrm{v}$", ha="left", va="top", color=cwarm, bbox=bbox)
tk = np.arange(-2, 3) * np.pi
ax.set(xticks=tk, yticks=tk, xticklabels=[f"${pistr(v)}$" for v in tk],
       yticklabels=[f"${pistr(v)}$" for v in tk], xlim=(-lim, lim), ylim=(lim, -lim), aspect="equal",
       xlabel="$\\nu_\\mathrm{h}$", ylabel="$\\nu_\\mathrm{v}$")
plt.show()

複製どうしが重ならないこと（エリアシングが生じないこと）を確かめる。凸多角形どうしの共通部分を求める関数（Sutherland–Hodgman 法）と面積の公式（靴ひもの公式）をここで定義して使う。

In [ ]:
def clip_convex(subj, clip):
    """凸多角形 subj を凸多角形 clip（頂点は反時計回り）で切り取る（Sutherland–Hodgman 法）"""
    out = [np.asarray(p, dtype=float) for p in subj]
    for i in range(len(clip)):
        a, b = np.asarray(clip[i], dtype=float), np.asarray(clip[(i + 1) % len(clip)], dtype=float)
        side = lambda p: (b[0] - a[0]) * (p[1] - a[1]) - (b[1] - a[1]) * (p[0] - a[0])
        inp, out = out, []
        for j in range(len(inp)):
            p, q = inp[j], inp[(j + 1) % len(inp)]
            sp, sq = side(p), side(q)
            if sp >= 0:
                out.append(p)
            if sp * sq < 0:
                out.append(p + (q - p) * sp / (sp - sq))
        if not out:
            break
    return np.array(out)


def area(poly):
    """多角形の面積（靴ひもの公式）"""
    if len(poly) < 3:
        return 0.0
    x, y = poly[:, 0], poly[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


ovl = 0
for mv in range(-1, 2):
    for mh in range(-1, 2):
        if mv == 0 and mh == 0:
            continue
        s = Lp @ [mv, mh]
        ovl += area(clip_convex(sup, sup + [s[1], s[0]]))
print("ovl =", ovl)                            # 0 なら重ならない

## ダウンサンプラとアップサンプラ
間引き $y[n]=x[Mn]$ と零値挿入（各サンプルの間に $M-1$ 個の零を挿入）。$M=2$ の例：

In [ ]:
xd = np.array([2, 2, 4, 2, 2, 4])
yds = xd[::2]
xu = np.array([2, 2, 4])
yus = np.zeros(2 * xu.size, dtype=int)
yus[::2] = xu
print("yds =", yds)
print("yus =", yus)
assert np.array_equal(yds, [2, 4, 2]) and np.array_equal(yus, [2, 0, 2, 0, 4, 0])   # スライドの値

## レート変換器（デシメータ・インタポレータ）の数値例
デシメータ：平均フィルタ $h[n]=(\frac12,\frac12)$（$n=0,1$）の後に 2 対 1 の間引き。

In [ ]:
v = signal.lfilter([1 / 2, 1 / 2], 1, xd)      # v[n] = (x[n]+x[n-1])/2（x[-1]=0）
ydec = v[1::2]                                 # 平均が揃う奇数番目を出力（位相の選択）
print("v =", v)
print("ydec =", ydec)
assert np.array_equal(v, [1, 2, 3, 3, 2, 3]) and np.array_equal(ydec, [2, 3, 3])   # スライドの値

インタポレータ：零値挿入の後，最近傍補間フィルタ $f[m]=(1,1)$（$m=0,1$）。

In [ ]:
yint = signal.lfilter([1.0, 1.0], 1, yus)
print("yint =", yint)
assert np.array_equal(yint, [2, 2, 2, 2, 4, 4])   # スライドの値

## エリアシング（間引き率 2）
間引き後のスペクトルは $Y(\mathrm{e}^{\mathrm{j}\omega})=\frac12X(\mathrm{e}^{\mathrm{j}\omega/2})+\frac12X(\mathrm{e}^{\mathrm{j}(\omega-2\pi)/2})$。帯域 $0.7\pi$ の信号では，引き伸ばされた 2 つの成分が重なる。通過域 $|\omega|<\pi/2$ の理想低域通過フィルタの出力 $v[n]$ を間引けば重ならない。

In [ ]:
w = np.linspace(-np.pi, 3 * np.pi, 2001)


def Xw(om):
    """周期 2π，帯域 0.7π"""
    return np.maximum(1 - np.abs(np.mod(om + np.pi, 2 * np.pi) - np.pi) / (0.7 * np.pi), 0)


def Hid(om):
    """理想低域通過（π/2）"""
    return (np.abs(np.mod(om + np.pi, 2 * np.pi) - np.pi) < np.pi / 2).astype(float)


def Vw(om):
    return Hid(om) * Xw(om)


Y1 = 0.5 * (Xw(w / 2) + Xw((w - 2 * np.pi) / 2))
fig, axs = plt.subplots(3, 1, figsize=(5.6, 4.4), sharex=True, layout="constrained")
axs[0].plot(w, Xw(w), color=ccool, lw=2)
axs[0].set_title(r"$X(\mathrm{e}^{\mathrm{j}\omega})$")
axs[1].plot(w, 0.5 * Xw(w / 2), "--", color=cmain, lw=1.5)
axs[1].plot(w, 0.5 * Xw((w - 2 * np.pi) / 2), "--", color=cgray, lw=1.5)
axs[1].plot(w, Y1, color=cwarm, lw=2)
axs[1].set_title(r"$Y(\mathrm{e}^{\mathrm{j}\omega})$")
axs[1].text(-0.95 * np.pi, 0.68, "フィルタなしで間引き：破線の 2 成分が重なる", color=cwarm, va="center", fontsize=8)
axs[2].plot(w, 0.5 * Vw(w / 2), "--", color=cmain, lw=1.5)
axs[2].plot(w, 0.5 * Vw((w - 2 * np.pi) / 2), "--", color=cgray, lw=1.5)
axs[2].set_title(r"$\frac{1}{2}V(\mathrm{e}^{\mathrm{j}\omega/2})+\frac{1}{2}V(\mathrm{e}^{\mathrm{j}(\omega-2\pi)/2})$")
axs[2].text(-0.95 * np.pi, 0.68, "低域通過フィルタの後に間引き：重ならない", color=cmain, va="center", fontsize=8)
axs[2].set_xlabel(r"$\omega$")
for ax in axs:
    ax.set(xticks=np.arange(-1, 4) * np.pi, xlim=(-np.pi, 3 * np.pi))
    ax.grid(True)
axs[2].set_xticklabels([r"$-\pi$", "$0$", r"$\pi$", r"$2\pi$", r"$3\pi$"])
axs[0].set(ylim=(0, 1.2), yticks=[0, 1])
for ax in axs[1:]:
    ax.set(ylim=(0, 0.8), yticks=[0, 0.5])
plt.show()

## イメージング（補間率 2）
零値挿入後のスペクトルは $Y(\mathrm{e}^{\mathrm{j}\omega})=X(\mathrm{e}^{\mathrm{j}2\omega})$。$\omega=\pi$ のまわりに余分な成分（イメージング）が現れる。補間フィルタ（理想低域通過，利得 2）で取り除く。

In [ ]:
def Xw2(om):
    return np.maximum(1 - np.abs(np.mod(om + np.pi, 2 * np.pi) - np.pi) / (0.8 * np.pi), 0)


Yi = Xw2(2 * w)
Fi = 2 * Hid(w)
fig, axs = plt.subplots(3, 1, figsize=(6.4, 4.6), sharex=True, layout="constrained")
axs[0].plot(w, Xw2(w), color=ccool, lw=2)
axs[0].set_title(r"$X(\mathrm{e}^{\mathrm{j}\omega})$")
axs[1].plot(w, Yi, color=cwarm, lw=2)
axs[1].set_title(r"$Y(\mathrm{e}^{\mathrm{j}\omega})=X(\mathrm{e}^{\mathrm{j}2\omega})$")
axs[1].text(0.5 * np.pi, 1.8, "零値挿入後：イメージングが生じる", color=cwarm, ha="center", va="center", fontsize=8)
axs[2].plot(w, Fi * Yi, color=cmain, lw=2)
axs[2].set_title(r"$F(\mathrm{e}^{\mathrm{j}\omega})Y(\mathrm{e}^{\mathrm{j}\omega})$")
axs[2].text(np.pi, 1.75, "イメージング除去", color=cmain, ha="center", va="center", fontsize=8)
axs[2].set_xlabel(r"$\omega$")
for ax in axs:
    ax.set(ylim=(0, 2.3), xticks=np.arange(-1, 4) * np.pi, xlim=(-np.pi, 3 * np.pi))
    ax.grid(True)
axs[2].set_xticklabels([r"$-\pi$", "$0$", r"$\pi$", r"$2\pi$", r"$3\pi$"])
plt.show()

## デシメーションフィルタと補間フィルタの振幅応答
平均フィルタ $(\frac12,\frac12)$：$|H|=|\cos(\omega/2)|$。最近傍補間フィルタ $(1,1)$：$|F|=2|\cos(\omega/2)|$。線形補間フィルタ $(\frac12,1,\frac12)$：$F=1+\cos\omega$。

凡例の代わりに，曲線のそばに名前を添える。

In [ ]:
wp = np.linspace(0, np.pi, 501)
fig, ax = plt.subplots(figsize=(4.8, 3.6), layout="constrained")
ax.plot(wp, (wp < np.pi / 2).astype(float), "--", color=cgray, lw=1.5)
ax.plot(wp, np.abs(np.cos(wp / 2)), color=cmain, lw=2)
ax.grid(True)
ax.set(xlim=(0, np.pi), ylim=(0, 1.15))
ax.text(0.03 * np.pi, 1.07, "理想特性", color=cgray, va="center", fontsize=9)
ax.text(0.6 * np.pi, 0.98, "平均値フィルタ\n(1/2,1/2)", color=cmain, va="top", fontsize=9)
ax.set(xticks=[0, np.pi / 2, np.pi], xticklabels=["$0$", r"$\pi/2$", r"$\pi$"],
       xlabel=r"$\omega$", ylabel=r"$|H(\mathrm{e}^{\mathrm{j}\omega})|$")
plt.show()

補間フィルタ。$\omega=\pi$（イメージングの中心）では最近傍も線形も 0。$\omega=3\pi/4$ での値を丸印で示す。

In [ ]:
vals = np.array([2 * abs(np.cos(3 * np.pi / 8)), 1 + np.cos(3 * np.pi / 4)])
print("vals =", vals)
print(f"F(3π/4)：最近傍 {vals[0]:.2f}，線形 {vals[1]:.2f}")
assert (f"{vals[0]:.2f}", f"{vals[1]:.2f}") == ("0.77", "0.29")   # スライドの値
fig, ax = plt.subplots(figsize=(4.8, 3.6), layout="constrained")
ax.plot(wp, 2 * (wp < np.pi / 2).astype(float), "--", color=cgray, lw=1.5)
ax.plot(wp, 2 * np.abs(np.cos(wp / 2)), color=cwarm, lw=2)
ax.plot(wp, 1 + np.cos(wp), color=cmain, lw=2)
ax.plot(3 * np.pi / 4, vals[0], "o", color=cwarm, mfc="w", mew=1.5)
ax.plot(3 * np.pi / 4, vals[1], "o", color=cmain, mfc="w", mew=1.5)
ax.grid(True)
ax.set(xlim=(0, np.pi), ylim=(0, 2.35))
ax.text(0.03 * np.pi, 2.18, "理想特性（利得 2）", color=cgray, va="center", fontsize=9)
ax.text(0.62 * np.pi, 1.45, "最近傍 (1,1)", color=cwarm, va="center", fontsize=9)
ax.text(0.47 * np.pi, 0.8, "線形 (1/2,1,1/2)", color=cmain, ha="right", va="center", fontsize=9)
ax.set(xticks=[0, np.pi / 2, 3 * np.pi / 4, np.pi], xticklabels=["$0$", r"$\pi/2$", r"$3\pi/4$", r"$\pi$"],
       xlabel=r"$\omega$", ylabel=r"$|F(\mathrm{e}^{\mathrm{j}\omega})|$")
plt.show()

## 二次元の補間フィルタの周波数応答
教科書の例題「補間フィルタ」：補間率 $2\times2$ 倍の零次ホールドフィルタ $F_\mathrm{ZH}(\boldsymbol{z})=(1+z_\mathrm{v}^{-1})(1+z_\mathrm{h}^{-1})$ と双線形補間フィルタ $F_\mathrm{BL}(\boldsymbol{z})=\frac14(z_\mathrm{v}+2+z_\mathrm{v}^{-1})(z_\mathrm{h}+2+z_\mathrm{h}^{-1})$ の利得を，インパルス応答から直接計算する。

In [ ]:
fzh = np.ones((2, 2))                          # 零次ホールド（n_v, n_h = 0,1）
fbl = np.outer([1, 2, 1], [1, 2, 1]) / 4       # 双線形（n_v, n_h = -1,0,1）


def Fzh(wv, wh):
    n = np.arange(2)
    return np.sum(fzh * np.exp(-1j * (wv * n[:, None] + wh * n[None, :])))


def Fbl(wv, wh):
    n = np.arange(-1, 2)
    return np.sum(fbl * np.exp(-1j * (wv * n[:, None] + wh * n[None, :])))


gainDC = np.round(np.abs([Fzh(0, 0), Fbl(0, 0)]), 10)                          # ω = 0（直流）：ともに M = |det(M)| = 4
gainPi2 = np.round(np.abs([Fzh(np.pi / 2, np.pi / 2), Fbl(np.pi / 2, np.pi / 2)]), 10)   # ω = (π/2 π/2)^T：零次ホールド 2，双線形 1
print("gainDC =", gainDC)
print("gainPi2 =", gainPi2)
assert np.array_equal(gainDC, [4, 4]) and np.array_equal(gainPi2, [2, 1])     # 教科書の解答と一致

## 3次畳み込み補間
教科書の例題「3次畳み込み補間」：Keys の補間核 $\kappa(q)$（$a=-1/2$）による補間率 $2\times2$ 倍の補間フィルタ $f[\boldsymbol{n}]=\kappa(n_1/2)\,\kappa(n_2/2)$，$\boldsymbol{n}\in\{-4,\ldots,4\}^2$。

In [ ]:
a = -1 / 2


def kappa(q):
    q = np.abs(q)
    return (((a + 2) * q**3 - (a + 3) * q**2 + 1) * (q <= 1)
            + (a * q**3 - 5 * a * q**2 + 8 * a * q - 4 * a) * ((q > 1) & (q <= 2)))


nn = np.arange(-4, 5)
f1 = kappa(nn / 2)                             # 一次元の補間フィルタ f_d[n_d]
print("f1 =", f1)
fcub = np.outer(f1, f1)                        # 二次元（分離型）

教科書の解答の行列（小数第 4 位）と比べる。

In [ ]:
ftext = np.array([
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0.0039, 0, -0.0352, -0.0625, -0.0352, 0, 0.0039, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, -0.0352, 0, 0.3164, 0.5625, 0.3164, 0, -0.0352, 0],
    [0, -0.0625, 0, 0.5625, 1.0000, 0.5625, 0, -0.0625, 0],
    [0, -0.0352, 0, 0.3164, 0.5625, 0.3164, 0, -0.0352, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0.0039, 0, -0.0352, -0.0625, -0.0352, 0, 0.0039, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0]])
assert np.array_equal(np.round(fcub, 4), ftext)   # 教科書と一致

スライドには中央の行 $f[0,n_2]$（$n_2=-4,\ldots,4$）を載せる。零は "0"，それ以外は小数第 4 位まで（教科書の書式）。

In [ ]:
row = ["0" if abs(v) < 1e-12 else f"{v:.4f}" for v in fcub[4]]
print(",".join(row))
assert ",".join(row) == "0,-0.0625,0,0.5625,1.0000,0.5625,0,-0.0625,0"   # スライドの値

## 画像の縮小（分離処理）
垂直・水平とも間引き率 2。フィルタなし，平均 $(\frac12,\frac12)$，$(\frac14,\frac12,\frac14)$ を比べる。

偶数長のフィルタ $(\frac12,\frac12)$ は原点を第 0 要素に置き（`origin=-1`），右隣・下隣との平均をとる。

In [ ]:
h2 = np.array([1 / 2, 1 / 2])
h3 = np.array([1 / 4, 1 / 2, 1 / 4])
D0 = X[::2, ::2]
D1 = ndimage.correlate(X, np.outer(h2, h2), mode="nearest", origin=-1)[::2, ::2]
D2 = ndimage.correlate(X, np.outer(h3, h3), mode="nearest")[::2, ::2]
fig, axs = plt.subplots(1, 3, figsize=(10, 3.6), layout="constrained")
show(axs[0], D0, "フィルタなし")
show(axs[1], D1, "(1/2,1/2)")
show(axs[2], D2, "(1/4,1/2,1/4)")
plt.show()

## まとめ
- 標本化は周波数スペクトルを周期化する（一次元は $2\pi/\Delta_\mathrm{t}$，多次元は周期構造行列 $2\pi\boldsymbol{L}^{-\top}$）。間隔が粗いとエリアシングが生じる
- 縮小は「低域通過フィルタ＋間引き」（デシメーション），拡大は「零値挿入＋補間フィルタ」（インタポレーション）
- 平均による縮小と双線形補間による拡大は，それぞれエリアシングとイメージングを抑える

© Copyright, Shogo MURAMATSU, All rights reserved.